In [11]:
### Import necessary libraries and modules

import os
import sys
import json

from dotenv import load_dotenv
from pydantic import BaseModel, Field

from typing import List, Dict, Any, Optional, Annotated
from typing_extensions import TypedDict

from langchain_openai import ChatOpenAI

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

from langchain_core.messages import BaseMessage,HumanMessage

from langgraph.checkpoint.memory import MemorySaver

In [12]:
## Define the structure of the chatbot state using TypedDict
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage],add_messages]

In [13]:
## Load environment variables from .env file
load_dotenv()

True

In [14]:
## Load the environment variable for the OpenAI API key
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")


In [15]:
model = ChatOpenAI(model="gpt-4o-mini", openai_api_key=OPENAI_API_KEY)

In [16]:
def chat_node(state: ChatState) -> ChatState:
    messages = state["messages"]
    response = model.invoke(messages)
    return {"messages": [response]}

In [17]:
config = {
    "configurable": {
        "thread_id": "1"
    }
}

In [18]:
## Define the checkpoint saver
checkpointer = MemorySaver()

In [19]:
graph = StateGraph(ChatState)


graph.add_node("chat_node", chat_node)
graph.add_edge(START, "chat_node")
graph.add_edge("chat_node",END)


chatbot = graph.compile(checkpointer=checkpointer)

In [24]:
result = chatbot.invoke(
    {
        "messages": [
            HumanMessage(content="What is my name?")
        ]
    },
    config=config
)

print(result["messages"][-1].content)

Your name is Mohammad Shuaib.
